# RQ3 — MP Co-signing Network + Influence

**Algorithms (syllabus mapping)**
- W5 PageRank, HITS — MP centrality scoring (Spark GraphFrames or NetworkX)
- W11 Louvain — community detection
- W11 Betweenness — bridge MPs
- W12 UMAP — MP embedding visualisation

**Inputs**: `silver_cosign_edges` (mp_guid, onerge_guid) + `silver_mp_party`.

**Outputs (Gold)**:
- `mp_pagerank` — MP centrality scores
- `mp_community` — Louvain community per MP
- `mp_embedding` — 2D coords

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'src'))
from spark_utils import get_spark, read_delta, write_delta, TABLES, GOLD
from pyspark.sql import functions as F
import networkx as nx
import pandas as pd

spark = get_spark('rq3', memory='6g')
spark.sparkContext.setLogLevel('WARN')
edges = read_delta(spark, TABLES['silver_cosign_edges'])
mp = read_delta(spark, TABLES['silver_mp_party'])
print('edges:', edges.count(), 'MPs:', mp.count())

## 1. Build co-signing graph

Two MPs share an edge if they co-signed at least one önerge. Weight = count of co-signed önergeler.

In [ ]:
a = edges.alias('a')
b = edges.alias('b')
pair = (
    a.join(b, F.col('a.onerge_guid') == F.col('b.onerge_guid'))
     .filter(F.col('a.mp_guid') < F.col('b.mp_guid'))
     .groupBy(F.col('a.mp_guid').alias('src'), F.col('b.mp_guid').alias('dst'))
     .agg(F.count('*').alias('weight'))
)
pair_pd = pair.toPandas()
print(f'co-sign edges: {len(pair_pd)}')
pair_pd.head()

## 2. NetworkX graph + PageRank + Louvain

In [ ]:
G = nx.Graph()
for _, row in pair_pd.iterrows():
    G.add_edge(row['src'], row['dst'], weight=int(row['weight']))
print(f'nodes: {G.number_of_nodes()}, edges: {G.number_of_edges()}')

pr = nx.pagerank(G, weight='weight')
bc = nx.betweenness_centrality(G, weight='weight', normalized=True)
try:
    import community as community_louvain  # python-louvain
except ImportError:
    !pip install -q python-louvain
    import community as community_louvain
partition = community_louvain.best_partition(G, weight='weight', random_state=42)

mp_pd = mp.toPandas().set_index('mp_guid')
ranking = pd.DataFrame({
    'mp_guid': list(G.nodes()),
    'pagerank': [pr[n] for n in G.nodes()],
    'betweenness': [bc[n] for n in G.nodes()],
    'community': [partition[n] for n in G.nodes()],
}).merge(mp_pd, left_on='mp_guid', right_index=True, how='left')
ranking.sort_values('pagerank', ascending=False).head(15)

## 3. Community ↔ party cross-tab

In [ ]:
xtab = pd.crosstab(ranking['community'], ranking['party'])
print(xtab)
purity = xtab.max(axis=1).sum() / xtab.values.sum()
print(f'Community purity vs party: {purity:.3f}')

## 4. UMAP 2D embedding (spectral fallback if umap not installed)

In [ ]:
try:
    import umap
    from sklearn.preprocessing import StandardScaler
    # adjacency-based feature vec
    A = nx.to_numpy_array(G, weight='weight')
    A = StandardScaler().fit_transform(A)
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(A)
except ImportError:
    pos = nx.spring_layout(G, weight='weight', seed=42, k=0.15)
    nodes = list(G.nodes())
    emb = [[pos[n][0], pos[n][1]] for n in nodes]
    import numpy as np
    emb = np.array(emb)

ranking['x'] = emb[:, 0]
ranking['y'] = emb[:, 1]
ranking.head()

## 5. Write Gold

In [ ]:
ranking_spark = spark.createDataFrame(ranking)
edges_spark = spark.createDataFrame(pair_pd)
write_delta(ranking_spark, GOLD / 'mp_ranking')
write_delta(edges_spark, GOLD / 'mp_edges')
print('Gold written.')